# 自动求导


---

## 1. 例子 1：标量情形

### 假设
- 向量：$\mathbf{x}, \mathbf{w}\in\mathbb{R}^n$  
- 标量：$y\in\mathbb{R}$

定义：
$$
z = \big(\langle \mathbf{x}, \mathbf{w} \rangle - y\big)^2
$$

### 目标
求关于 $\mathbf{w}$ 的梯度 $\dfrac{\partial z}{\partial \mathbf{w}}$。

### 分解（把复杂运算拆为简单操作）
令
$$
\begin{aligned}
a &= \langle \mathbf{x}, \mathbf{w} \rangle \quad\text{（点积）}\\
b &= a - y\\
z &= b^2
\end{aligned}
$$

### 应用链式法则
链式法则给出：
$$
\frac{\partial z}{\partial \mathbf{w}}
= \frac{\partial z}{\partial b}\cdot\frac{\partial b}{\partial a}\cdot\frac{\partial a}{\partial \mathbf{w}}
$$

局部导数为：
- $\dfrac{\partial z}{\partial b} = 2b$
- $\dfrac{\partial b}{\partial a} = 1$
- $\dfrac{\partial a}{\partial \mathbf{w}} = \mathbf{x}^T$

因此：
$$
\boxed{\;
\frac{\partial z}{\partial \mathbf{w}} = 2\big(\langle \mathbf{x}, \mathbf{w}\rangle - y\big)\mathbf{x}^T
\;}
$$

**直观说明**：该梯度是平方损失对权重的梯度；方向与 $\mathbf{x}$ 相同，幅度与误差成正比。

---

## 2. 例子 2：向量 / 矩阵情形

### 假设
- $\mathbf{X}\in\mathbb{R}^{m\times n}$，$\mathbf{w}\in\mathbb{R}^n$，$\mathbf{y}\in\mathbb{R}^m$。  
定义：
$$
z = \|\mathbf{Xw} - \mathbf{y}\|^2 = (\mathbf{Xw}-\mathbf{y})^\top(\mathbf{Xw}-\mathbf{y})
$$

### 分解
令
$$
a = \mathbf{Xw},\qquad b = a - \mathbf{y},\qquad z=\|b\|^2
$$

### 求导（链式法则）
$$
\frac{\partial z}{\partial \mathbf{w}}
= \frac{\partial z}{\partial b}\cdot\frac{\partial b}{\partial a}\cdot\frac{\partial a}{\partial \mathbf{w}}
$$

局部项：
- $\dfrac{\partial z}{\partial b} = 2 b^\top$
- $\dfrac{\partial b}{\partial a} = I_{m}$
- $\dfrac{\partial a}{\partial \mathbf{w}} = \mathbf{X}$

结果：
$$
\boxed{\;
\frac{\partial z}{\partial \mathbf{w}}
= 2(\mathbf{Xw}-\mathbf{y})^\top \mathbf{X}
\;}
$$

> 备注：若希望梯度以列向量形式表示，可写成 $\nabla_{\mathbf{w}} z = 2\mathbf{X}^\top(\mathbf{Xw}-\mathbf{y})$。

**直观**：这是最小二乘（OLS）问题的常见梯度表达式。

---

## 3. 计算图（Computation Graph）

### 概念
- 把整个计算拆成一系列基础运算（节点），用有向无环图 (DAG) 表示依赖关系。
- 节点保存中间变量，反向传播时利用这些中间值计算局部导数并累积全局梯度。

### 对例子 1 的分解（示意）
输入：$\mathbf{x},\mathbf{w},y$  
中间：$a=\langle\mathbf{x},\mathbf{w}\rangle,\; b=a-y$  
输出：$z=b^2$

文本结构示意：

- 前向：从叶节点计算到根节点并保存中间结果（a, b）。
- 反向：从 z 开始，按链式法则向下传播梯度。

---

## 4. 自动求导的两种模式

### 链式法则（数学基础）
若 $y$ 是多层复合函数的结果，则
$$
\frac{dy}{dx} = \frac{dy}{du_n}\cdot\frac{du_n}{du_{n-1}}\cdots\frac{du_1}{dx}.
$$

### 正向累积（Forward Mode）
- 计算方向：**从输入到输出**。
- 适合：**输入维度小、输出维度大** 的场景。
- 思路：正向传播时同时维护每个中间变量关于目标输入的导数。

### 反向累积（Reverse Mode / 反向传播）
- 计算方向：**从输出到输入**。
- 适合：**输入维度大、输出维度小**（例如深度学习，一个标量损失对大量参数求梯度）。
- 思路：前向计算并存储中间值；反向从输出开始按依赖倒序计算并累积局部梯度。

---

## 5. 反向累积示例（一步步演示）

以 $z=(\langle x,w\rangle - y)^2$ 为例：

### 前向（计算并保存）
- $a=\langle x,w\rangle$
- $b=a-y$
- $z=b^2$

保存 $a,b$ 供反向使用。

### 反向（从输出往回）
- $\dfrac{\partial z}{\partial b} = 2b$
- $\dfrac{\partial z}{\partial a} = \dfrac{\partial z}{\partial b}\cdot\dfrac{\partial b}{\partial a} = 2b\cdot 1 = 2b$
- $\dfrac{\partial z}{\partial w} = \dfrac{\partial z}{\partial a}\cdot\dfrac{\partial a}{\partial w} = 2b\cdot x$

最终：
$$
\frac{\partial z}{\partial w} = 2(\langle x,w\rangle - y)\,x.
$$

---

## 6. 反向累积总结（实践要点）
1. 构造计算图并确定依赖顺序。  
2. 前向执行时保存必要的中间变量（不要保存冗余数据）。  
3. 反向执行时从输出遍历回输入，按链式法则累积梯度。  
4. 可对与目标无关的分支进行剪枝以节省计算。

---

## 7. 复杂度分析

- **时间复杂度**：反向传播通常为 $O(n)$，$n$ 为操作/节点数。正向与反向代价通常相近（反向常数倍更高）。
- **空间复杂度**：反向模式需保存前向中间结果，典型为 $O(n)$。
- **适用场景对比表**：

| 模式 | 时间复杂度 | 空间复杂度 | 适用场景 |
|---|---:|---:|---|
| 正向累积 | $O(n)$ | 低（或 $O(1)$） | 输入少，输出多 |
| 反向累积 | $O(n)$ | $O(n)$ | 输入多，输出少（如神经网络） |



In [1]:
import torch
x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

In [2]:
x.requires_grad_(True)
x.grad

In [3]:
y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

In [4]:
y.backward()
x.grad

tensor([ 0.,  4.,  8., 12.])

In [5]:
x.grad == 4 * x

tensor([True, True, True, True])

In [6]:
#在默认情况下，pytorch会积累梯度，我们需要清除之前的值
x.grad.zero_()
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])